In [0]:
# Databricks notebook source
# %run ./00_config

# ---- Configuration ----
CATALOG = "dbacademy"
SCHEMA  = "labuser13955035_1772876383"
VOL     = f"/Volumes/{CATALOG}/{SCHEMA}/h2files"

# ---- ENTSO-E Load ----
for year in [2020, 2021]:
    tbl = f"{CATALOG}.{SCHEMA}.load_{year}_raw"
    src = f"{VOL}/entsoe_nl_{year}_load.csv"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {tbl} USING DELTA AS
        SELECT
          `MTU (CET/CEST)` AS mtu_cet_cest,
          Area AS area,
          `Actual Total Load (MW)` AS actual_load_mw,
          `Day-ahead Total Load Forecast (MW)` AS forecast_load_mw,
          input_file_name() AS source_file,
          current_timestamp() AS ingested_at_utc
        FROM read_files('{src}', format => 'csv', header => true)
    """)
    print(f"✅ {tbl}")

# ---- ENTSO-E Generation ----
for year in [2020, 2021]:
    tbl = f"{CATALOG}.{SCHEMA}.generation_{year}_raw"
    src = f"{VOL}/entsoe_nl_{year}_generation.csv"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {tbl} USING DELTA AS
        SELECT
          `MTU (CET/CEST)` AS mtu_cet_cest,
          Area AS area,
          `Production Type` AS production_type,
          `Generation (MW)` AS generation_mw,
          input_file_name() AS source_file,
          current_timestamp() AS ingested_at_utc
        FROM read_files('{src}', format => 'csv', header => true)
    """)
    print(f"✅ {tbl}")

# ---- ENTSO-E Prices ----
for year in [2020, 2021]:
    tbl = f"{CATALOG}.{SCHEMA}.prices_{year}_raw"
    src = f"{VOL}/entsoe_nl_{year}_prices.csv"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {tbl} USING DELTA AS
        SELECT
          `MTU (CET/CEST)` AS mtu_cet_cest,
          Area AS area,
          Sequence AS sequence,
          `Day-ahead Price (EUR/MWh)` AS dayahead_price_eur_mwh,
          `Intraday Period (CET/CEST)` AS intraday_period_cet_cest,
          `Intraday Price (EUR/MWh)` AS intraday_price_eur_mwh,
          input_file_name() AS source_file,
          current_timestamp() AS ingested_at_utc
        FROM read_files('{src}', format => 'csv', header => true)
    """)
    print(f"✅ {tbl}")

print("\n✅ ENTSO-E ingest complete")

In [0]:
# ---- ERA5 Weather ----
for year in [2020, 2021]:
    tbl = f"{CATALOG}.{SCHEMA}.weather_{year}_raw"
    src = f"{VOL}/era5_nl_{year}_hourly.csv"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {tbl} USING DELTA AS
        SELECT *, input_file_name() AS source_file, current_timestamp() AS ingested_at_utc
        FROM read_files('{src}', format => 'csv', header => true)
    """)
    print(f"✅ {tbl}")

print("\n✅ ERA5 weather ingest complete")


In [0]:
# ---- H2 Stations ----
tbl = f"{CATALOG}.{SCHEMA}.h2_raw_h2stations"
src = f"{VOL}/h2stations_nl.csv"
spark.sql(f"""
    CREATE OR REPLACE TABLE {tbl} USING DELTA AS
    SELECT *, input_file_name() AS source_file, current_timestamp() AS ingested_at_utc
    FROM read_files('{src}', format => 'csv', header => true)
""")
print(f"✅ {tbl}")
print("\n✅ All bronze ingest complete!")


In [0]:
# ---- Day 1: Delta Optimization ----
# Demonstrate OPTIMIZE (compacts small files) and version history

raw_tables = [
    f"{CATALOG}.{SCHEMA}.load_2020_raw",
    f"{CATALOG}.{SCHEMA}.load_2021_raw",
    f"{CATALOG}.{SCHEMA}.prices_2020_raw",
    f"{CATALOG}.{SCHEMA}.prices_2021_raw",
    f"{CATALOG}.{SCHEMA}.generation_2020_raw",
    f"{CATALOG}.{SCHEMA}.generation_2021_raw",
    f"{CATALOG}.{SCHEMA}.weather_2020_raw",
    f"{CATALOG}.{SCHEMA}.weather_2021_raw",
    f"{CATALOG}.{SCHEMA}.h2_raw_h2stations",
]

for tbl in raw_tables:
    spark.sql(f"OPTIMIZE {tbl}")
    print(f"✅ OPTIMIZED {tbl.split('.')[-1]}")

# Show version history for one table (demonstrates Delta time travel)
print("\n📜 Version History (prices_2020_raw):")
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{SCHEMA}.prices_2020_raw"))